# Traffic validation: rust vs. DSim (Berlin v7.1)

Compares the traffic of the rust simulation (matsim-rust) with Java DSim from the trips files of both engines
(`output_trips.csv.zst` of rust, `<runId>.output_trips.csv.zst` of DSim). Loading and figures are in
`src/main/python/validation/`, so this notebook and
`cd src/main/python && uv run --group analysis python -m validation --rust <output dir> --dsim <output dir>`
(all figures at once) use the same definitions. Figures go to `plots/rust-vs-dsim-validation/` (gitignored).

Start Jupyter with `uv run --group analysis jupyter lab`.

**Runs**
- *it0*: the mobsim only, with the calibrated plans, so both engines simulate the same plans and every trip can be
  compared on its own (joined on `trip_id`). Grid: traffic {queue, kinematicWaves} x link {FIFO, PassingQ} x pct
  {1, 3, 10}.
- *it100*: 100 iterations with 3 seeds at 1 pct; plans differ between runs, so compare distributions there.
- Locally the runs are in `output/validation/<run>/output/`; from the cluster in runs-svn
  `~/runs-svn/matsim-rust/iterations-vs-dsim-2026/<experiment>/runs/<run_id>/output/` (rust experiments
  `validation-*`, DSim `dsim-validation-*`, read with `read_experiment`).

**Definitions**
- Travel time `trav_time` and departure time `dep_time` in seconds (HH:MM:SS in the file, also beyond 24 h);
  `traveled_distance` in meters.
- Per-trip comparison: run *a* (rust) against reference *b* (DSim), joined on `trip_id`; Δ = a - b. Modes are the
  `main_mode` of the trip; a trip whose main mode differs between the runs only counts in *all*.


In [ ]:
%config InlineBackend.figure_format = 'retina'
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from validation.load import DATA_ROOT, read_experiment, read_trips, runs_of
from validation.plots import agreement, departures, departures_per_hour, save, trav_time_ecdf, trip_scatter


def show(fig, name):
    """Saves the figure as PDF and PNG and shows it."""
    paths = save(fig, name)
    display(fig)
    plt.close(fig)
    return paths

## Runs

The local it0 pair at 1 pct with pt (traffic queue, link FIFO). For runs from the cluster, use `read_experiment` (next cell) and pick the runs with `runs_of`.

In [ ]:
validation = project_root.parents[2] / "output/validation"
rust = read_trips(validation / "rust-1pct-pt-queue-fifo/output")
dsim = read_trips(validation / "dsim-1pct-pt-queue-fifo/output")

pd.DataFrame({name: t["main_mode"].value_counts() for name, t in
              {"rust": rust, "dsim": dsim}.items()}).fillna(0).astype(int)

In [ ]:
# Experiments from the cluster (runs-svn), if fetched: one frame per run, e.g.
#   trips = read_experiment("validation-it0") ; dsim_trips = read_experiment("dsim-validation-it0")
#   rust = runs_of(trips, pct=1, traffic="queue", link="FIFO")  # {run_id: trips}
sorted(p.name for p in DATA_ROOT.glob("*validation*"))

## Metric 1: departures per hour

Caption: *Departures per hour over the day (0-36 h) per main mode, rust solid and DSim dashed (top), and the
difference rust - DSim (bottom). Truck and freight share one color.*

In [ ]:
show(departures(rust, dsim), "departures")

In [ ]:
# The same as table (rust - DSim per hour), the hours with any difference
difference = departures_per_hour(rust) - departures_per_hour(dsim)
difference[(difference != 0).any(axis=1)]

## Metric 2: travel time and distance per trip

Caption: *Per trip, the value in rust (y) over the value in DSim (x), joined on `trip_id`, one panel per main mode:
hexagons with a logarithmic count, the dashed line is 1:1. Both axes are logarithmic, so trips with 0 in one of the
runs are not drawn but counted in the summary: share of identical values and Pearson r (agreement of single trips),
mean of Δ = rust - DSim with its share of the DSim mean (systematic deviation, in which deviations in both directions
cancel, e.g. +5 % = 5 % more vehicle hours in rust), and median of Δ (deviation of a typical trip).*

`agreement` gives the numbers per mode: trips, share identical, median and mean of rust - DSim, the mean relative to the
DSim mean (`rel_mean_diff`), median |rust - DSim| (seconds or meters; how much single trips scatter) and r. Its `attrs`
count the trips only in one of the runs.

Known difference of the car distances (2026-11-11): rust's experienced plans have no `distance` on network routes, so
MATSim's reader sets it without the start and the end link (`RouteUtils.calcDistanceExcludingStartEndLink`), while
Java's `EventsToLegs` counts the end link. The routes are the same; rust's distances lack the length of the end link.

In [ ]:
stats = agreement(rust, dsim)
print(stats.attrs)
stats.set_index(["column", "mode"]).round({"identical": 4, "median_diff": 1, "mean_diff": 1, "rel_mean_diff": 4,
                                          "median_abs_diff": 1, "r": 5})

In [ ]:
show(trip_scatter(rust, dsim, "trav_time"), "scatter_trav_time")

In [ ]:
show(trip_scatter(rust, dsim, "traveled_distance"), "scatter_traveled_distance")

In [ ]:
# Trips in only one of the runs
only_rust = rust[~rust["trip_id"].isin(dsim["trip_id"])]
only_dsim = dsim[~dsim["trip_id"].isin(rust["trip_id"])]
pd.concat({"only rust": only_rust["main_mode"].value_counts(), "only dsim": only_dsim["main_mode"].value_counts()},
          axis=1).fillna(0).astype(int)

## Metric 3: distribution of the travel time

Caption: *Distribution (ECDF) of the travel time per main mode, rust and DSim, logarithmic travel time. Δ median is
the median of rust minus that of DSim (of the distributions, not trip by trip). KS D is the Kolmogorov-Smirnov
distance: the largest vertical gap between the two curves, i.e. the largest difference of the shares of trips below
some travel time (0 = same distribution, 0.02 = at most 2 percentage points); it has no sign.*

In [ ]:
show(trav_time_ecdf(rust, dsim), "trav_time_ecdf")

In [ ]:
# Quantiles of the travel time in minutes
quantiles = pd.concat({name: t.groupby("main_mode")["trav_time"].quantile([0.1, 0.5, 0.9]).unstack() / 60
                       for name, t in {"rust": rust, "dsim": dsim}.items()}, axis=1)
quantiles.swaplevel(axis=1).sort_index(axis=1).round(2)